# Creating the `LIB` table

From the `books_df` that was created in a different notebook I will contruct a `LIB` table.

## Import Packages

In [106]:
import pandas as pd
import re

## Set the Data Path

In [107]:
file_location = 'C:/Users/mason/Box/Text As Data Final/Output'
books_df = pd.read_csv(file_location + '/BrandonSandersonBooks.csv')
books_df.head()

,title,file_path,author,date,chapter_id,paragraph_id,text
0,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,0,by Robert Jordan
1,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,1,The Eye of the World
2,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,2,The Great Hunt
3,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,3,The Dragon Reborn
4,A Memory of Light,C:/Users/mason/Box/Text As Data Final/Sanderso...,Robert Jordan & Brandon Sanderson,2013-01-08,0,4,The Shadow Rising


## Correct the chapter_id

I don't want all of the beginning cruft but I do want the prologue for each book as chapter 0

In [108]:
start_pattern = r'^[\W_]*(?:Prologue|Preface|Introduction|Foreword|Chapter 1|Chapter One|Book One|Book 1|Part One|Part 1|The Wheel of Time turns|GOD.S DEATH|YOU WANT|Eshonai|Yeah. he.dheard|Nomad woke up|In the middle|Right about then|Elantris was beautiful|Starling hopped|ASH FELL FROM THE SKY|Lord Tresting frowned)\b'
end_pattern = r'^\s*(?:Endnotes|About the Author|Glossary|Ars Arcanum|Postscript|POSTSCRIPT|Epilogue|EPILOGUE|Afterword|Also by Brandon Sanderson|Illustrations:|THE END)\b'

books_df['temp_word_count'] = books_df['text'].str.split().str.len()
books_df['is_short_line'] = books_df['temp_word_count'] < 10
books_df['short_density'] = books_df.groupby('file_path')['is_short_line'].transform(
    lambda x: x.rolling(window=15, center=True, min_periods=1).sum()
)
is_not_toc = books_df['short_density'] < 12

raw_is_start = books_df['text'].str.contains(start_pattern, flags=re.IGNORECASE, regex=True, na=False)

books_df['is_start'] = raw_is_start & is_not_toc
books_df['has_started'] = books_df.groupby('file_path')['is_start'].cumsum()

book_has_any_start = books_df.groupby('file_path')['is_start'].transform('max')

valid_start_mask = (books_df['has_started'] > 0) | (book_has_any_start == 0) & (books_df['chapter_id']> 0)

raw_is_end = books_df['text'].str.contains(end_pattern, flags=re.IGNORECASE, regex=True, na=False)
books_df['is_end'] = raw_is_end & valid_start_mask & is_not_toc
books_df['has_ended'] = books_df.groupby('file_path')['is_end'].cumsum()

valid_end_mask = (books_df['has_ended'] == 0)

clean_books_df = books_df[valid_start_mask & valid_end_mask].copy()

clean_books_df = clean_books_df.drop(columns=['is_start', 'is_end', 'has_started', 'has_ended', 'is_short_line', 'short_density', 'temp_word_count'])
clean_books_df = clean_books_df.reset_index(drop=True)

print(f"Original number of rows: {len(books_df)}")
print(f"Cleaned number of rows: {len(clean_books_df)}")

LIB_titles = clean_books_df['title'].unique()
print(f"Unique book titles in the cleaned dataset ({len(LIB_titles)}):")
print(LIB_titles)

Original number of rows: 287646
Cleaned number of rows: 86466
Unique book titles in the cleaned dataset (24):
<StringArray>
[             'A Memory of Light',              'Arcanum Unbounded',
                       'Elantris',         'Isles of the Emberdark',
                    'Oathbringer',                  'Rhythm of War',
                'Shadows of Self',                     'Steelheart',
            'The Aether of Night',               'The Alloy of Law',
          'The Bands of Mourning',               'The Final Empire',
               'The Hero of Ages',                 'The Lost Metal',
                 'The Sunlit Man',               'The Way of Kings',
          'The Well of Ascension',              'The Wheel of Time',
             'Towers of Midnight',       'Tress of the Emerald Sea',
                     'Warbreaker',                 'Wind and Truth',
              'Words of Radiance', 'Yumi and the Nightmare Painter']
Length: 24, dtype: str


## Still getting wierd Chapter Patterns lets clean that up

In [109]:
number_words = r"One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten|Eleven|Twelve|Thirteen|Fourteen|Fifteen|Sixteen|Seventeen|Eighteen|Nineteen|Twenty|Thirty|Forty|Fifty|Sixty|Seventy|Eighty|Ninety|Hundred"

chapter_pattern = rf'^\s*(?:Prologue|Epilogue|Preface|Introduction|Foreword|Chapter|Interlude|Part|Book)\b|^\s*\d{{1,3}}[.:-]?\s*$|^\s*(?:{number_words}|-)+\s*$'

clean_books_df['is_new_chapter'] = clean_books_df['text'].str.contains(
    chapter_pattern, flags=re.IGNORECASE, regex=True, na=False)

clean_books_df['logical_chapter_id'] = clean_books_df.groupby('file_path')['is_new_chapter'].cumsum()

## Diagnostic View of each Title

In [110]:
diagnostic_df = clean_books_df.copy()
diagnostic_df['text_preview'] = diagnostic_df['text'].astype(str).str[:100].str.replace('\n', ' ')+'...'


first_few = diagnostic_df.groupby('title').head(5)
last_few = diagnostic_df.groupby('title').tail(5)


boundaries_df = pd.concat([first_few, last_few]).drop_duplicates().sort_values(['title', 'logical_chapter_id'])

## Get Book Length, number of chapters, Number of Paragraphs

In [111]:
clean_books_df['word_count'] = clean_books_df['text'].str.split().str.len()

clean_books_df['is_real_paragraph'] = clean_books_df['word_count'] > 3

valid_text_df = clean_books_df[clean_books_df['is_real_paragraph']]
chapter_counts = valid_text_df.groupby('file_path')['logical_chapter_id'].nunique().reset_index(name='Total_chapters')

lib_df = clean_books_df.groupby(
    ['title', 'file_path', 'author', 'date'],
    dropna=False
).agg(
    length_in_words=('word_count', 'sum'),
    Total_paragraphs=('is_real_paragraph', 'sum')
).reset_index()

lib_df = lib_df.merge(chapter_counts, on='file_path', how='left')

lib_df = lib_df.rename(columns={
    'file_path': 'File_path',
    'author': 'Author',
    'date': 'Date',
    'length_in_words': 'Length',
})

lib_df = lib_df[['title', 'Author', 'Date', 'Length', 'Total_chapters', 'Total_paragraphs', 'File_path']]
lib_df = lib_df.sort_values('title').reset_index(drop=True)

clean_books_df = clean_books_df.drop(columns=['word_count', 'is_real_paragraph', 'is_new_chapter'])

with pd.option_context('display.max_rows', None, 'display.max_columns', 1000):
    print(lib_df)

                             title                             Author  \
0                A Memory of Light  Robert Jordan & Brandon Sanderson   
1                Arcanum Unbounded                  Brandon Sanderson   
2                         Elantris                  Brandon Sanderson   
3           Isles of the Emberdark                  Brandon Sanderson   
4                      Oathbringer                  Brandon Sanderson   
5                    Rhythm of War                  Brandon Sanderson   
6                  Shadows of Self                  Brandon Sanderson   
7                       Steelheart                  Brandon Sanderson   
8              The Aether of Night                 Sanderson, Brandon   
9                 The Alloy of Law                  Brandon Sanderson   
10           The Bands of Mourning                 Sanderson, Brandon   
11                The Final Empire                  Brandon Sanderson   
12                The Hero of Ages                 

## Fix the Dates Column

In [112]:
lib_df['Date'] = pd.to_datetime(lib_df['Date'], errors='coerce', utc=True)
lib_df.loc[lib_df['Date'].dt.year < 1900, 'Date'] = pd.NaT
lib_df['Date'] = lib_df['Date'].dt.tz_localize(None)
#lib_df['Date'] = lib_df['Date'].dt.strftime('%Y-%m-%d')
print(lib_df[['title', 'Date']])

                             title       Date
0                A Memory of Light 2013-01-08
1                Arcanum Unbounded        NaT
2                         Elantris        NaT
3           Isles of the Emberdark        NaT
4                      Oathbringer        NaT
5                    Rhythm of War        NaT
6                  Shadows of Self 2015-11-06
7                       Steelheart        NaT
8              The Aether of Night        NaT
9                 The Alloy of Law        NaT
10           The Bands of Mourning        NaT
11                The Final Empire        NaT
12                The Hero of Ages        NaT
13                  The Lost Metal        NaT
14                  The Sunlit Man 2023-10-01
15                The Way of Kings        NaT
16           The Well of Ascension 2007-08-21
17               The Wheel of Time        NaT
18              Towers of Midnight        NaT
19        Tress of the Emerald Sea        NaT
20                      Warbreaker

In [113]:
missing_dates = lib_df[lib_df['Date'].isna()]
print("Books with missing or invalid dates:")
print(missing_dates[['title', 'Date']])

Books with missing or invalid dates:
                       title Date
1          Arcanum Unbounded  NaT
2                   Elantris  NaT
3     Isles of the Emberdark  NaT
4                Oathbringer  NaT
5              Rhythm of War  NaT
7                 Steelheart  NaT
8        The Aether of Night  NaT
9           The Alloy of Law  NaT
10     The Bands of Mourning  NaT
11          The Final Empire  NaT
12          The Hero of Ages  NaT
13            The Lost Metal  NaT
15          The Way of Kings  NaT
17         The Wheel of Time  NaT
18        Towers of Midnight  NaT
19  Tress of the Emerald Sea  NaT
21            Wind and Truth  NaT
22         Words of Radiance  NaT


In [114]:
found_dates = {
    'Arcanum Unbounded': '2016-11-22',
    'Elantris': '2005-04-21',
    'Isles of the Emberdark': '2025-01-01',
    'Oathbringer': '2017-11-14',
    'Rhythm of War': '2020-11-17',
    'Steelheart': '2013-09-24',
    'The Aether of Night': '2003-01-01', 
    'The Alloy of Law': '2011-11-08',
    'The Bands of Mourning': '2016-01-26',
    'The Final Empire': '2006-07-17',
    'The Hero of Ages': '2008-10-14',
    'The Lost Metal': '2022-11-15',
    'The Way of Kings': '2010-08-31',
    'The Wheel of Time': '1990-01-15',
    'Towers of Midnight': '2010-11-02',
    'Tress of the Emerald Sea': '2023-01-01',
    'Wind and Truth': '2024-12-06',
    'Words of Radiance': '2014-03-04'
}

replacement_dates = lib_df['title'].map(found_dates)
replacement_dates = pd.to_datetime(replacement_dates)
lib_df['Date'] = lib_df['Date'].fillna(replacement_dates)
lib_df['Date'] = lib_df['Date'].dt.strftime('%Y-%m-%d')
print(lib_df[['title', 'Date']])

                             title        Date
0                A Memory of Light  2013-01-08
1                Arcanum Unbounded  2016-11-22
2                         Elantris  2005-04-21
3           Isles of the Emberdark  2025-01-01
4                      Oathbringer  2017-11-14
5                    Rhythm of War  2020-11-17
6                  Shadows of Self  2015-11-06
7                       Steelheart  2013-09-24
8              The Aether of Night  2003-01-01
9                 The Alloy of Law  2011-11-08
10           The Bands of Mourning  2016-01-26
11                The Final Empire  2006-07-17
12                The Hero of Ages  2008-10-14
13                  The Lost Metal  2022-11-15
14                  The Sunlit Man  2023-10-01
15                The Way of Kings  2010-08-31
16           The Well of Ascension  2007-08-21
17               The Wheel of Time  1990-01-15
18              Towers of Midnight  2010-11-02
19        Tress of the Emerald Sea  2023-01-01
20           

In [115]:
print(lib_df)

                             title                             Author  \
0                A Memory of Light  Robert Jordan & Brandon Sanderson   
1                Arcanum Unbounded                  Brandon Sanderson   
2                         Elantris                  Brandon Sanderson   
3           Isles of the Emberdark                  Brandon Sanderson   
4                      Oathbringer                  Brandon Sanderson   
5                    Rhythm of War                  Brandon Sanderson   
6                  Shadows of Self                  Brandon Sanderson   
7                       Steelheart                  Brandon Sanderson   
8              The Aether of Night                 Sanderson, Brandon   
9                 The Alloy of Law                  Brandon Sanderson   
10           The Bands of Mourning                 Sanderson, Brandon   
11                The Final Empire                  Brandon Sanderson   
12                The Hero of Ages                 

In [116]:
author_corrections ={
    'Sanderson, Brandon': 'Brandon Sanderson',
    'Unkown Author': 'Robert Jordan & Brandon Sanderson'
}
lib_df['Author'] = lib_df['Author'].replace(author_corrections)
print(lib_df[['title', 'Author']])

                             title                             Author
0                A Memory of Light  Robert Jordan & Brandon Sanderson
1                Arcanum Unbounded                  Brandon Sanderson
2                         Elantris                  Brandon Sanderson
3           Isles of the Emberdark                  Brandon Sanderson
4                      Oathbringer                  Brandon Sanderson
5                    Rhythm of War                  Brandon Sanderson
6                  Shadows of Self                  Brandon Sanderson
7                       Steelheart                  Brandon Sanderson
8              The Aether of Night                  Brandon Sanderson
9                 The Alloy of Law                  Brandon Sanderson
10           The Bands of Mourning                  Brandon Sanderson
11                The Final Empire                  Brandon Sanderson
12                The Hero of Ages                  Brandon Sanderson
13                  

In [117]:
lib_df.head()

,title,Author,Date,Length,Total_chapters,Total_paragraphs,File_path
0,A Memory of Light,Robert Jordan & Brandon Sanderson,2013-01-08,279300,1,8694,C:/Users/mason/Box/Text As Data Final/Sanderso...
1,Arcanum Unbounded,Brandon Sanderson,2016-11-22,30328,2,946,C:/Users/mason/Box/Text As Data Final/Sanderso...
2,Elantris,Brandon Sanderson,2005-04-21,199771,1,6372,C:/Users/mason/Box/Text As Data Final/Sanderso...
3,Isles of the Emberdark,Brandon Sanderson,2025-01-01,35349,1,1212,C:/Users/mason/Box/Text As Data Final/Sanderso...
4,Oathbringer,Brandon Sanderson,2017-11-14,117420,1,4002,C:/Users/mason/Box/Text As Data Final/Sanderso...


## Upload and Save the `LIB` table
We did drop the Total_Chapters and Total_Paragrpahs Columns as they were inconsistent when pulling from the epub formatting

In [118]:
lib_df.to_csv(file_location + '/BrandonSanderson_LIB.csv', index=False)